# 02 — Cache frozen-backbone embeddings (Phase 3, Kaggle T4)

Runs the **expensive** step ONCE per asset: load TimesFM 2.5 with
`torch_compile=False`, freeze the backbone, and cache the last-patch embedding +
ReVIN stats + normalized 1-step target for every rolling window. Heads then train
fast on these caches (notebook 03) with no backbone forward.

Reads cleaned returns from the mounted `tsfm-clean-data` Kaggle Dataset (built
locally via `wrds_data.build_clean_dataset`). Add it under *Add Data* first.

## 1. Install + clone (then RESTART KERNEL)

In [ ]:
!pip install "git+https://github.com/google-research/timesfm.git#egg=timesfm[torch]" -q
!pip install "tsfm_cal @ git+https://github.com/YOURNAME/tsfm-thesis.git" -q
# >>> Restart kernel, then continue. <<<

## 2. Load native module (torch_compile=False) + freeze backbone

In [ ]:
import numpy as np, torch
from tsfm_cal import config, data, backbone, io_utils

wrapper, native = backbone.load_native(torch_compile=False)
backbone.freeze_backbone(native)
trainable = sum(p.requires_grad for p in native.parameters())
print("backbone frozen; trainable backbone tensors:", trainable, "(expect 0)")

## 3. Cache embeddings for every asset (saves npz per asset)

In [ ]:
for key in config.ASSET_KEYS:
    label = config.asset_display(key)
    try:
        _dates, rets = data.load_returns(key, kind=config.RETURNS_TYPE)
    except Exception as e:
        print(f"! skip {key}: {e}"); continue
    if len(rets) < config.CONTEXT + 50:
        print(f"! skip {key}: only {len(rets)} returns"); continue
    print(f"Caching {label} ({key}), n={len(rets)} ...")
    cache = backbone.cache_embeddings(native, rets, context=config.CONTEXT,
                                      batch_size=256, label=label)
    out = backbone.save_cache(key, cache)
    print(f"  saved {out}  emb={cache['emb'].shape}")

## 4. Sanity-check one cache

In [ ]:
c = backbone.load_cache("SPY")
print({k: v.shape for k, v in c.items()})
print("target_norm mean/std:", c["target_norm"].mean(), c["target_norm"].std())

## 5. Zip caches for download / re-use in notebook 03

In [ ]:
import shutil, os
root = io_utils.base_dir() / "finetune"
zip_path = shutil.make_archive("/kaggle/working/emb_caches", "zip", root)
print("Download:", zip_path)
for r,_,fs in os.walk(root):
    for f in fs: print(os.path.join(r,f))